# 09 — Running the same pipeline on two different datasets

The whole point of this project was that I shouldn't have to rewrite the code
when the dataset changes. This notebook is where I actually prove that — same
`RetailPipeline.from_files(path).run()` call on Rossmann (1.02M rows, 1,115
stores, daily aggregates) and Walmart (8.4K rows, transactional order-lines,
no store column).

I want to see:
1. Does the pipeline finish on both without me intervening?
2. What does the smart-inference layer decide for each dataset?
3. How do the actual metrics compare?


> **Note** — outputs in this notebook are not pre-rendered. Open it in Jupyter / VS Code / Colab and **Run All** to see results. The full Rossmann pipeline takes ~1–2 min locally.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt


## Setup — one function that runs the whole thing

In [ ]:
from retailmind import RetailPipeline, ask, save_report

def run_dataset(main_path, aux=None, label=''):
    print(f'--- {label} ---')
    p = RetailPipeline.from_files(main_path, auxiliary_paths=aux)
    p.horizon = 14
    p.max_entities_for_full_forecast = 5
    p.canonicalize_()
    p.eda_()
    p.forecast_(sample_entities=30, cv_folds=2)
    p.anomalies_(sample_entities=10)
    p.drivers_(sample_entities=30)
    p.recommendations_()
    return p

## Rossmann

In [ ]:
ros = run_dataset('../train.csv', aux=['../store.csv'], label='Rossmann')
print()
print('What the pipeline auto-decided:')
print(ros.decision_log.render())
print()
print('Forecast CV metrics:')
print(json.dumps(ros.forecast_model.cv_metrics['mean'], indent=2))

On Rossmann the pipeline mostly just ran — no major smart-inference
decisions because the columns already had names like `Date`, `Store`, `Sales`
that the mapper recognises. SMAPE around 19% on a 14-day forecast, and the
seasonal-naive baseline is at 44%, so the model is doing real work.


## Walmart

In [ ]:
wal = run_dataset('../walmart Retail Data.xlsx', label='Walmart')
print()
print('What the pipeline auto-decided:')
print(wal.decision_log.render())
print()
print('Forecast CV metrics:')
print(json.dumps(wal.forecast_model.cv_metrics['mean'], indent=2))

Walmart is more interesting because it has no `Store` column at all.
The smart-inference layer noticed this and promoted `Region` to entity_id,
so we now get per-region forecasts instead of one global series. The absolute
metrics are way worse than Rossmann (SMAPE ~87% vs 19%) — but the baseline
is even worse at 106%, so the model is still adding value. The reason
Walmart is hard: only 1,460 daily rows once you aggregate, and the rows are
just transactions that happened to land on different days, not real store-day
sales like Rossmann.


## Side-by-side scorecard

In [ ]:
def card(p, name):
    m = p.forecast_model.cv_metrics['mean']
    dr = p.driver_report
    return {
        'dataset': name,
        'canonical_rows': len(p.canonical),
        'entities': p.canonical['entity_id'].nunique(),
        'aggregation': p.chosen_freq,
        'forecast_smape': round(m.get('smape', 0), 1),
        'baseline_smape': round(m.get('baseline_smape', 0), 1),
        'rmse_lift_pct': round(m.get('rmse_lift_pct', 0), 1),
        'driver_r2': round(dr.r2, 3),
        'baseline_r2': round(dr.baseline_r2 or 0, 3),
        'anomalies': len(p.anomalies) if p.anomalies is not None else 0,
        'reorder_entities': len(p.recommendations) if p.recommendations is not None else 0,
    }

pd.DataFrame([card(ros, 'Rossmann'), card(wal, 'Walmart')])

## Ask the same question of both pipelines

In [ ]:
for p, name in [(ros, 'Rossmann'), (wal, 'Walmart')]:
    print(f'--- {name} ---')
    print(ask(p, 'How accurate is the forecast?', mode='offline'))
    print()

## Generate the static HTML reports (for GitHub Pages)

In [ ]:
ros_path = save_report(ros, out_dir='../docs', filename='rossmann.html', dataset_name='Rossmann Store Sales')
wal_path = save_report(wal, out_dir='../docs', filename='walmart.html', dataset_name='Walmart Retail Data')
print(f'Rossmann report: {ros_path}')
print(f'Walmart report : {wal_path}')

## What I think this proves

Same code ran on two pretty different datasets. The smart-inference layer
made dataset-specific decisions (auto-promoting Region for Walmart) without
me hard-coding anything. Metrics are honest — Walmart is genuinely hard and
the numbers show that. To prove this on a third dataset I'd just call
`RetailPipeline.from_files('something_else.csv').run()` and the same scorecard
table would populate.

For the Streamlit version where you can upload your own file see `app.py`.
For the test suite that confirms both datasets still pass after every change
see `tests/test_universal.py`.
